generator libs

In [1]:
from dataset_generators.valid_bpd_datasets import(
 generate_adversarial_dataset,
 generate_bnpl_dataset,
 generate_calibration_dataset,
 generate_churn_dataset,
 generate_credit_scoring_dataset,
 generate_delivery_failure_dataset,
 generate_disparate_impact_dataset,
 generate_employee_attrition_dataset,
 generate_equalized_odds_dataset,
 generate_fraud_dataset,
 generate_insurance_claim_dataset,
 generate_medical_diagnosis_dataset,
 generate_phishing_url_dataset,
 generate_predictive_maintenance_dataset,
 generate_proxy_dataset,
 generate_purchase_dataset,
 generate_recruitment_dataset,
 generate_spam_dataset,
 generate_ng_fintech_dataset
)
from dataset_generator_utils import DatasetBundle

Dataset generators

In [2]:
DATASET_GENERATORS = {
    "fraud": generate_fraud_dataset,
    "credit_scoring": generate_credit_scoring_dataset,
    "churn": generate_churn_dataset,
    "insurance": generate_insurance_claim_dataset,
    "medical": generate_medical_diagnosis_dataset,
    "attrition": generate_employee_attrition_dataset,
    "purchase": generate_purchase_dataset,
    "recruitment": generate_recruitment_dataset,
    "spam": generate_spam_dataset,
    "phishing": generate_phishing_url_dataset,
    "maintenance": generate_predictive_maintenance_dataset,
    "delivery": generate_delivery_failure_dataset,
    "bnpl": generate_bnpl_dataset,
    "ng_fintech": generate_ng_fintech_dataset,
    "proxy": generate_proxy_dataset,
    "disparate_impact": generate_disparate_impact_dataset,
    "equalized_odds": generate_equalized_odds_dataset,
    "calibration": generate_calibration_dataset,
    "adversarial": generate_adversarial_dataset,
}

training the model

In [ ]:
import os
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from dataset_generator_utils import train_test_validation_split
# model directory
MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

def train_model(bundle: DatasetBundle, name: str) -> dict:
    """
    Train a logistic regression model with train/val/test split.
    Save the trained model to disk and return metrics.
    """
    data_split = train_test_validation_split(bundle)

    X_train, y_train = data_split['train'].X, data_split['train'].y
    X_val, y_val = data_split['validation'].X, data_split['validation'].y
    X_test, y_test = data_split["test"].X, data_split["test"].y

    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)

    preds_val = model.predict(X_val)
    preds_test = model.predict(X_test)

    val_acc = accuracy_score(y_val, preds_val)
    test_acc = accuracy_score(y_test, preds_test)

    # Save model
    model_path = os.path.join(MODEL_DIR, f"{name}_model.pkl")
    joblib.dump(model, model_path)

    return {
        "dataset": name,
        "train_rows": len(X_train),
        "val_rows": len(X_val),
        "test_rows": len(X_test),
        "features": list(X_train.columns),
        "val_accuracy": val_acc,
        "test_accuracy": test_acc,
        "model_path": model_path,
        "metadata": bundle.metadata,
    }


In [ ]:
import os
import pandas as pd
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, as_completed

def optimal_workers(num_urls: int) -> int:
    """
    Dynamically choose a safe number of workers.
    - At least 2
    - At most CPU count * 2
    - Never more than number of URLs
    """
    cpu_count = os.cpu_count() or multiprocessing.cpu_count() or 4
    return max(2, min(num_urls, cpu_count * 2))


def train_all_datasets():
    results = []

    max_workers = optimal_workers(len(DATASET_GENERATORS))
    print(f"\nUsing {max_workers} workers for {len(DATASET_GENERATORS)} datasets...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_name = {
            executor.submit(train_model, gen(), name): name
            for name, gen in DATASET_GENERATORS.items()
        }

        for number, future in enumerate(as_completed(future_to_name), start=1):
            name = future_to_name[future]
            print(f"\n[{number}/{len(future_to_name)}] Training on {name} dataset...")
            try:
                result = future.result()
                results.append(result)
                print(f"{name} done | Val Acc: {result['val_accuracy']:.3f} | Test Acc: {result['test_accuracy']:.3f}")
                print(f"Model saved at {result['model_path']}")
            except Exception as e:
                print(f"{name} failed: {e}")
                results.append({"dataset": name, "status": "error", "error": str(e)})

    return pd.DataFrame(results)